# Feature Engineering & Preprocessing Pipeline

Implements Section 5 of `MultiTask_MLP_ChurnLTV_ProjectPlan_v2.pdf` (Data Pipeline & Feature Engineering): merges `train`, `members`, `transactions`, and `user_logs` into one row per user, engineers the LTV target and behavioral features, then encodes/scales/splits the data so it's ready for the PyTorch `Dataset` in the next notebook.

**Design decisions that depart from the plan's literal wording (documented as we hit them):**
- **LTV target leakage fix.** The first version of this pipeline defined `ltv` as the *full-history* sum of `actual_amount_paid`, while also including `num_transactions` and `avg_actual_amount_paid` (both full-history aggregates of the same transactions) as input features. Since `num_transactions * avg_actual_amount_paid == ltv` almost exactly by construction, the LTV head could reconstruct its target algebraically instead of learning anything (Exp-2 baseline hit R²=0.999, which was the tell). The fix: a hard **temporal split**. `FEATURE_CUTOFF` (2016-12-31) is the last date any *feature* is allowed to see. `ltv` is redefined as **forward-looking revenue** — the sum of `actual_amount_paid` in the `LTV_TARGET_WINDOW` (2017-01-01 to 2017-02-28) — which by construction cannot be reconstructed from pre-cutoff aggregates. This also happens to be a more realistic LTV framing for the business use case in Section 8 (near-term revenue at risk, not historical revenue already collected).
- `daily_active_days` (and the other `user_logs` engagement features) are computed over the **30 days immediately before `FEATURE_CUTOFF`** (2016-12-02 to 2016-12-31), not the full 2015-2017 history and not the 30 days before the overall data cutoff — moving them earlier was necessary once `FEATURE_CUTOFF` was introduced, so no feature window overlaps the LTV target window. The plan's own feature table (Section 2.2) documents this column's range as "1-30", which only makes sense for a recent window — a lifetime count over `user_logs.csv` (392M rows, ~2 years) would range into the hundreds.
- `count(distinct date)` per user over the full 392M-row `user_logs` table was tested two ways: an exact `GROUP BY` didn't finish in a reasonable time (huge per-group hash-set cardinality), and DuckDB's approximate `approx_count_distinct` (HyperLogLog) finished in ~49s but produced implausible results (9% of users exceeded the theoretical maximum). Restricting to a 30-day window shrinks the scan enough to make the **exact** count trivial.
- `is_auto_renew_rate` and `avg_song_completion` are left unscaled (they're already bounded to [0,1]) even though the plan's architecture diagram (Section 4.1) labels all 9 numerical columns "StandardScaled" — the plan's own feature table (Section 2.2) marks these two as "Already 0-1", which is the more sensible choice and is what we follow here.

In [1]:
import json
import os

import duckdb
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = os.path.join(os.getcwd(), "data", "processed")
con = duckdb.connect()


def p(name):
    return os.path.join(PROCESSED_DIR, f"{name}.parquet")


# All input features may only see data on/before FEATURE_CUTOFF. The LTV target is
# forward-looking revenue in LTV_TARGET_WINDOW, which starts the day after FEATURE_CUTOFF
# — so no feature can leak information about the target.
FEATURE_CUTOFF = pd.Timestamp("2016-12-31")
FEATURE_CUTOFF_INT = 20161231
LTV_TARGET_START = 20170101
LTV_TARGET_END = 20170228
ENGAGEMENT_WINDOW_START = 20161202  # 30 days ending at FEATURE_CUTOFF
ENGAGEMENT_WINDOW_END = FEATURE_CUTOFF_INT

## 5.1 Data merging strategy

Base: `train.parquet` (992,931 rows — the original WSDM round-1 labels). Left-joined with `members`, transaction feature aggregates (restricted to `transaction_date <= FEATURE_CUTOFF`), and the 30-day pre-cutoff engagement window from `user_logs`. The LTV target itself is aggregated separately from a *later*, non-overlapping window (see below). Using `train`/`transactions`/`user_logs` (not the `_v2` refresh files) because their row counts match the plan's Section 2.1 table exactly (992,931 / 6.7M / 21.5M / 405M+).

In [2]:
merge_query = f"""
    with txn_agg as (
        -- feature aggregates: pre-cutoff transactions only
        select msno,
               count(*) as num_transactions,
               avg(payment_plan_days) as avg_payment_plan_days,
               avg(actual_amount_paid) as avg_actual_amount_paid,
               avg(is_auto_renew) as is_auto_renew_rate
        from '{p("transactions")}'
        where transaction_date <= {FEATURE_CUTOFF_INT}
        group by msno
    ),
    latest_txn as (
        -- most recent payment method as of the feature cutoff (not any later transaction)
        select msno, payment_method_id from (
            select msno, payment_method_id,
                   row_number() over (partition by msno order by transaction_date desc) rn
            from '{p("transactions")}'
            where transaction_date <= {FEATURE_CUTOFF_INT}
        ) where rn = 1
    ),
    ltv_target as (
        -- LTV target: forward-looking revenue strictly AFTER the feature cutoff
        select msno, sum(actual_amount_paid) as ltv
        from '{p("transactions")}'
        where transaction_date between {LTV_TARGET_START} and {LTV_TARGET_END}
        group by msno
    ),
    logs_agg as (
        -- total_secs clipped to [0, 86400]: a small fraction of rows have corrupted
        -- extreme values (see 01_EDA.ipynb, "Scanning the full user_logs history" section)
        select msno,
               count(distinct date) as daily_active_days,
               sum(greatest(least(total_secs, 86400), 0)) as total_secs_sum,
               sum(num_25) as sum25, sum(num_50) as sum50, sum(num_75) as sum75,
               sum(num_985) as sum985, sum(num_100) as sum100
        from '{p("user_logs")}'
        where date between {ENGAGEMENT_WINDOW_START} and {ENGAGEMENT_WINDOW_END}
        group by msno
    )
    select
        t.msno, t.is_churn,
        m.city, m.bd, m.gender, m.registered_via, m.registration_init_time,
        coalesce(lt_target.ltv, 0) as ltv,
        coalesce(txn.num_transactions, 0) as num_transactions,
        txn.avg_payment_plan_days,
        txn.avg_actual_amount_paid,
        coalesce(txn.is_auto_renew_rate, 0) as is_auto_renew_rate,
        lt.payment_method_id,
        coalesce(la.daily_active_days, 0) as daily_active_days,
        coalesce(la.total_secs_sum, 0) as total_secs_sum,
        coalesce(la.sum25, 0) as sum25, coalesce(la.sum50, 0) as sum50,
        coalesce(la.sum75, 0) as sum75, coalesce(la.sum985, 0) as sum985,
        coalesce(la.sum100, 0) as sum100
    from '{p("train")}' t
    left join '{p("members")}' m using (msno)
    left join txn_agg txn using (msno)
    left join latest_txn lt using (msno)
    left join ltv_target lt_target using (msno)
    left join logs_agg la using (msno)
"""
df = con.execute(merge_query).df()
print(f"merged shape: {df.shape}")
df.isna().sum()

merged shape: (992931, 20)


msno                           0
is_churn                       0
city                      115770
bd                        115770
gender                    601239
registered_via            115770
registration_init_time    115770
ltv                            0
num_transactions               0
avg_payment_plan_days      31554
avg_actual_amount_paid     31554
is_auto_renew_rate             0
payment_method_id          31554
daily_active_days              0
total_secs_sum                 0
sum25                          0
sum50                          0
sum75                          0
sum985                         0
sum100                         0
dtype: int64

Unlike the leaky full-history version, `avg_payment_plan_days`, `avg_actual_amount_paid`, and `payment_method_id` can now be null for the minority of users with no transaction on or before `FEATURE_CUTOFF` (2016-12-31) — e.g. someone who first subscribed in January 2017 (`num_transactions` and `is_auto_renew_rate` are `coalesce`d to 0 for these users directly in SQL). `city`/`bd`/`gender`/`registered_via`/`registration_init_time` are null for the ~11.7% of users absent from `members.parquet` entirely; `gender` is additionally null (missing but present in `members`) for many more, consistent with the ~65% gender-missing rate found in `01_EDA.ipynb`.

## 5.2 Feature engineering

In [3]:
# Registration tenure (days): FEATURE_CUTOFF - registration_init_time
reg_date = pd.to_datetime(
    df["registration_init_time"].astype("Int64").astype(str), format="%Y%m%d", errors="coerce"
)
df["registration_tenure_days"] = (FEATURE_CUTOFF - reg_date).dt.days

# Weighted song completion rate
song_totals = df[["sum25", "sum50", "sum75", "sum985", "sum100"]].sum(axis=1)
df["avg_song_completion"] = (
    0.25 * df["sum25"] + 0.50 * df["sum50"] + 0.75 * df["sum75"] + 0.985 * df["sum985"] + 1.0 * df["sum100"]
) / (song_totals + 1)

# Log listening time (30-day pre-cutoff window) and log-LTV target (forward-looking revenue)
df["total_secs_log"] = np.log1p(df["total_secs_sum"])
df["log1p_ltv"] = np.log1p(df["ltv"])

df[["registration_tenure_days", "avg_song_completion", "total_secs_log", "log1p_ltv"]].describe()

,registration_tenure_days,avg_song_completion,total_secs_log,log1p_ltv
count,877161.000000,992931.000000,992931.000000,992931.000000
mean,1215.982504,0.607833,8.266801,5.288436
std,1086.574728,0.371664,4.976228,0.990501
min,-89.000000,0.000000,0.000000,0.000000
25%,360.000000,0.000000,0.000000,5.293305
50%,962.000000,0.783065,10.732434,5.293305
75%,1787.000000,0.890417,11.823284,5.700444
max,4663.000000,0.999905,14.760742,7.084226


## 5.3 Preprocessing checklist

Order matters: split first, then fit every encoder/scaler/imputer statistic **only on the training split** to avoid leakage (per the plan's explicit instruction for `StandardScaler`, applied here consistently to every other fitted step too).

In [4]:
df["gender"] = df["gender"].fillna("unknown")
df["bd_clean"] = df["bd"].where(df["bd"].between(1, 100))  # invalid ages -> null, imputed after the split

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["is_churn"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["is_churn"], random_state=42)
splits = {"train": train_df, "val": val_df, "test": test_df}

for name, split in splits.items():
    print(f"{name:6s} n={len(split):>7,}  churn_rate={split['is_churn'].mean():.4f}")

train  n=695,051  churn_rate=0.0639
val    n=148,940  churn_rate=0.0639
test   n=148,940  churn_rate=0.0639


In [5]:
# Age: median imputation fit on train only
bd_median = train_df["bd_clean"].median()
# Registration tenure: median imputation for the ~11.7% of users absent from members.csv
tenure_median = train_df["registration_tenure_days"].median()
# Transaction aggregates: median imputation for users with no pre-cutoff transaction history
payment_plan_days_median = train_df["avg_payment_plan_days"].median()
actual_amount_paid_median = train_df["avg_actual_amount_paid"].median()
print(f"bd median (train): {bd_median}, tenure median (train): {tenure_median}")
print(f"avg_payment_plan_days median (train): {payment_plan_days_median}, "
      f"avg_actual_amount_paid median (train): {actual_amount_paid_median}")

for split in splits.values():
    split["bd_clean"] = split["bd_clean"].fillna(bd_median)
    split["registration_tenure_days"] = split["registration_tenure_days"].fillna(tenure_median)
    split["avg_payment_plan_days"] = split["avg_payment_plan_days"].fillna(payment_plan_days_median)
    split["avg_actual_amount_paid"] = split["avg_actual_amount_paid"].fillna(actual_amount_paid_median)

bd median (train): 28.0, tenure median (train): 964.0
avg_payment_plan_days median (train): 30.0, avg_actual_amount_paid median (train): 139.76923076923077


In [6]:
# Label-encode categoricals to integer indices starting at 0; an extra reserved index
# absorbs missing/unseen categories at val/test/inference time (nn.Embedding-safe).
CATEGORICAL_COLS = ["city", "gender", "registered_via", "payment_method_id"]
# Embedding dims per the plan's Section 3.1 heuristics
EMBEDDING_DIMS = {"city": 5, "gender": 2, "registered_via": 3, "payment_method_id": 8}


def fit_encoder(train_col):
    categories = sorted(train_col.dropna().unique().tolist())
    mapping = {cat: i for i, cat in enumerate(categories)}
    mapping["__unknown__"] = len(categories)
    return mapping


def apply_encoder(col, mapping):
    return col.map(mapping).fillna(mapping["__unknown__"]).astype(int)


encoders = {}
for col in CATEGORICAL_COLS:
    mapping = fit_encoder(train_df[col])
    encoders[col] = mapping
    for split in splits.values():
        split[f"{col}_enc"] = apply_encoder(split[col], mapping)
    print(f"{col:20s} cardinality (incl. unknown bucket): {len(mapping)}")

city                 cardinality (incl. unknown bucket): 22
gender               cardinality (incl. unknown bucket): 4


registered_via       cardinality (incl. unknown bucket): 6


payment_method_id    cardinality (incl. unknown bucket): 35


In [7]:
# StandardScaler on the 7 unbounded numerical columns, fit on train only.
# is_auto_renew_rate and avg_song_completion are already in [0, 1] and left unscaled.
SCALE_COLS = [
    "bd_clean", "registration_tenure_days", "avg_payment_plan_days",
    "avg_actual_amount_paid", "num_transactions", "total_secs_log", "daily_active_days",
]
UNSCALED_NUMERICAL_COLS = ["is_auto_renew_rate", "avg_song_completion"]

scaler = StandardScaler()
scaler.fit(train_df[SCALE_COLS])
for split in splits.values():
    split[[f"{c}_scaled" for c in SCALE_COLS]] = scaler.transform(split[SCALE_COLS])

train_df[[f"{c}_scaled" for c in SCALE_COLS]].describe()

,bd_clean_scaled,registration_tenure_days_scaled,avg_payment_plan_days_scaled,avg_actual_amount_paid_scaled,num_transactions_scaled,total_secs_log_scaled,daily_active_days_scaled
count,6.950510e+05,6.950510e+05,6.950510e+05,6.950510e+05,6.950510e+05,6.950510e+05,6.950510e+05
mean,-4.027817e-17,-1.009203e-16,-1.434884e-16,-1.275816e-16,3.598456e-18,4.849737e-17,-4.481713e-17
std,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00
min,-4.902477e+00,-1.245577e+00,-9.743942e-01,-9.924826e-01,-1.729166e+00,-1.661511e+00,-1.180998e+00
25%,-1.362265e-01,-7.632271e-01,-1.496330e-01,-3.379448e-01,-8.738304e-01,-1.661511e+00,-1.180998e+00
50%,-1.362265e-01,-2.174099e-01,-1.152679e-01,-6.839930e-02,1.036962e-01,4.956514e-01,-9.583341e-02
75%,-1.362265e-01,4.514359e-01,-1.152679e-01,-7.370136e-03,8.368412e-01,7.146401e-01,9.893310e-01
max,1.257378e+01,3.394357e+00,1.191250e+01,1.223050e+01,6.946383e+00,1.302902e+00,1.531913e+00


## Save model-ready splits, encoders, and feature manifest

Final feature set: 4 categorical + 9 numerical = 13 columns, matching the plan's Section 5.1 step-6 estimate ("~13 feature columns") exactly.

In [8]:
OUTPUT_COLS = (
    ["msno", "is_churn", "log1p_ltv", "ltv"]
    + [f"{c}_enc" for c in CATEGORICAL_COLS]
    + [f"{c}_scaled" for c in SCALE_COLS]
    + UNSCALED_NUMERICAL_COLS
)

for name, split in splits.items():
    out_path = os.path.join(PROCESSED_DIR, f"model_dataset_{name}.parquet")
    split[OUTPUT_COLS].to_parquet(out_path, engine="pyarrow", compression="zstd", index=False)
    print(f"wrote {out_path} ({len(split):,} rows, {len(OUTPUT_COLS)} cols)")

joblib.dump(encoders, os.path.join(PROCESSED_DIR, "categorical_encoders.joblib"))
joblib.dump(scaler, os.path.join(PROCESSED_DIR, "numerical_scaler.joblib"))

feature_manifest = {
    "feature_cutoff": str(FEATURE_CUTOFF.date()),
    "ltv_target_window": [LTV_TARGET_START, LTV_TARGET_END],
    "engagement_window": [ENGAGEMENT_WINDOW_START, ENGAGEMENT_WINDOW_END],
    "targets": {"churn": "is_churn", "ltv_log": "log1p_ltv", "ltv_raw": "ltv"},
    "categorical": {
        col: {
            "column": f"{col}_enc",
            "cardinality": len(encoders[col]),
            "embedding_dim": EMBEDDING_DIMS[col],
            "unknown_index": encoders[col]["__unknown__"],
        }
        for col in CATEGORICAL_COLS
    },
    "numerical_scaled": [f"{c}_scaled" for c in SCALE_COLS],
    "numerical_unscaled": UNSCALED_NUMERICAL_COLS,
}
with open(os.path.join(PROCESSED_DIR, "feature_manifest.json"), "w") as f:
    json.dump(feature_manifest, f, indent=2)

feature_manifest

wrote /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/model_dataset_train.parquet (695,051 rows, 17 cols)
wrote /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/model_dataset_val.parquet (148,940 rows, 17 cols)
wrote /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/model_dataset_test.parquet (148,940 rows, 17 cols)


{'feature_cutoff': '2016-12-31',
 'ltv_target_window': [20170101, 20170228],
 'engagement_window': [20161202, 20161231],
 'targets': {'churn': 'is_churn', 'ltv_log': 'log1p_ltv', 'ltv_raw': 'ltv'},
 'categorical': {'city': {'column': 'city_enc',
   'cardinality': 22,
   'embedding_dim': 5,
   'unknown_index': 21},
  'gender': {'column': 'gender_enc',
   'cardinality': 4,
   'embedding_dim': 2,
   'unknown_index': 3},
  'registered_via': {'column': 'registered_via_enc',
   'cardinality': 6,
   'embedding_dim': 3,
   'unknown_index': 5},
  'payment_method_id': {'column': 'payment_method_id_enc',
   'cardinality': 35,
   'embedding_dim': 8,
   'unknown_index': 34}},
 'numerical_scaled': ['bd_clean_scaled',
  'registration_tenure_days_scaled',
  'avg_payment_plan_days_scaled',
  'avg_actual_amount_paid_scaled',
  'num_transactions_scaled',
  'total_secs_log_scaled',
  'daily_active_days_scaled'],
 'numerical_unscaled': ['is_auto_renew_rate', 'avg_song_completion']}

## Verification

Reload the saved splits from disk and re-check shapes, churn-rate consistency, and null-freeness before treating this pipeline as done.

In [9]:
for name in ["train", "val", "test"]:
    check = pd.read_parquet(os.path.join(PROCESSED_DIR, f"model_dataset_{name}.parquet"))
    n_nulls = check.drop(columns=["msno"]).isna().sum().sum()
    print(
        f"{name:6s} shape={check.shape}  churn_rate={check['is_churn'].mean():.4f}  "
        f"ltv_median={check['ltv'].median():.0f}  total_nulls={n_nulls}"
    )

train  shape=(695051, 17)  churn_rate=0.0639  ltv_median=198  total_nulls=0
val    shape=(148940, 17)  churn_rate=0.0639  ltv_median=198  total_nulls=0
test   shape=(148940, 17)  churn_rate=0.0639  ltv_median=198  total_nulls=0
